<a href="https://colab.research.google.com/github/kaouchounesalah-eddine-ux/arabic-news-classification/blob/main/notebooks/05_araBERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Load dataset & Keep final test set

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Classroom', 'Document-WPS Office(1).pdf', 'ASS.pdf', 'WhatsApp Image 2024-09-17 at 13.10.46_7afe7ddc.jpg', 'kaouchoune.pdf', 'ffffffffffffffffff.pdf', 'I share the view that cycling is the best method of getting around cities and for a variety of.docx', 'SALAH EDDINE KAOUCHOUNE (1).pdf', 'analogiqueCR.pdf', 'SALAH EDDINE KAOUCHOUNE.pdf', 'analogique.pdf', 'Start research (3).gsheet', 'Start research (2).gsheet', 'Start research (1).gsheet', "juste translate this data to a tableau labels: ['....gsheet", "non non pourquoi tu cherche sur wikipidia ,j'ai d....gsheet", 'Start research.gsheet', 'SALAH-EDDINE KAOUCHOUNE Sec A.png', 'Google AI Studio', 'noname.pdf', 'Système d’exploitation Linux-IAGI-1-partie 2 (2).gdoc', 'Système d’exploitation Linux-IAGI-1-partie 2 (1).gdoc', 'Système d’exploitation Linux-IAGI-1-partie 2.gdoc', 'IMG_20260121_221731.pdf', 'Resume - Intern Software Developer.pdf', 'Données Formulaire : GET, POST, Sécurité.gsheet', "my weight is 88.6 kg and i'm 173 cm ,

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
# from preprocess import preprocess


# =========================
# 1. Load dataset
# =========================

import pandas as pd

df = pd.read_json(
    '/content/drive/MyDrive/articles.json'
)

X = df["body"]
y = df["categories"].str[0]


# =========================
# 2. Keep final test set
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


2. Development Data

In [4]:
print("=== DEVELOPMENT DATA ===")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nNumber of samples:", len(X_train))

print("\nCategory distribution:")
print(y_train.value_counts())

=== DEVELOPMENT DATA ===
X_train shape: (16800,)
y_train shape: (16800,)

Number of samples: 16800

Category distribution:
categories
ثقافة     2800
دولي      2800
اقتصاد    2800
رياضة     2800
سياسة     2800
مجتمع     2800
Name: count, dtype: int64


3. Preprocessing

4. Embedding

In [5]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)

print("Classes:")
for i, category in enumerate(label_encoder.classes_):
    print(i, "→", category)

Classes:
0 → اقتصاد
1 → ثقافة
2 → دولي
3 → رياضة
4 → سياسة
5 → مجتمع


In [6]:
from sklearn.model_selection import train_test_split

X_ft_train, X_ft_val, y_ft_train, y_ft_val = train_test_split(
    X_train,
    y_train_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_train_encoded
)

print("Training samples:", len(X_ft_train))
print("Validation samples:", len(X_ft_val))

Training samples: 13440
Validation samples: 3360


In [7]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": X_ft_train.tolist(),
    "labels": y_ft_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_ft_val.tolist(),
    "labels": y_ft_val.tolist()
})

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['text', 'labels'],
    num_rows: 13440
})
Dataset({
    features: ['text', 'labels'],
    num_rows: 3360
})


In [8]:
from transformers import AutoTokenizer

model_name = "aubmindlab/bert-base-arabertv02"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded.


In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=512
    )

In [10]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/13440 [00:00<?, ? examples/s]

Map:   0%|          | 0/3360 [00:00<?, ? examples/s]

In [11]:
print(train_tokenized)
print(val_tokenized)

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 13440
})
Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 3360
})


In [12]:
import numpy as np

sample_lengths = [
    len(tokenizer.encode(text, truncation=False))
    for text in X_ft_train.iloc[:1000]
]

print("Average tokens:", np.mean(sample_lengths))
print("Maximum tokens:", np.max(sample_lengths))
print("Median tokens:", np.median(sample_lengths))
print("Articles > 512 tokens:", sum(
    length > 512 for length in sample_lengths
))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (650 > 512). Running this sequence through the model will result in indexing errors


Average tokens: 262.067
Maximum tokens: 1506
Median tokens: 218.0
Articles > 512 tokens: 69


In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=6,
    id2label={
        i: category
        for i, category in enumerate(label_encoder.classes_)
    },
    label2id={
        category: i
        for i, category in enumerate(label_encoder.classes_)
    }
)

print("AraBERT classification model loaded.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  543MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AraBERT classification model loaded.


In [14]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [15]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(
        labels,
        predictions
    )

    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./arabert_news_classifier",

    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    learning_rate=1e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    logging_steps=100,

    report_to="none",

    fp16=True
)

In [17]:
import importlib.metadata

print("PEFT:", importlib.metadata.version("peft"))
print("Transformers:", importlib.metadata.version("transformers"))

PEFT: 0.20.0
Transformers: 5.16.1


In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Trainer ready.")

Trainer ready.


In [1]:
trainer.train()

NameError: name 'trainer' is not defined

In [ ]:
y_test_encoded = label_encoder.transform(y_test)

print("Number of test labels:", len(y_test_encoded))
print("First 10 encoded labels:", y_test_encoded[:10])

In [ ]:
label_encoder.transform(y_test)

In [ ]:
from datasets import Dataset

test_dataset = Dataset.from_dict({
    "text": X_test.tolist(),
    "labels": y_test_encoded.tolist()
})

print(test_dataset)

In [ ]:
test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

print(test_tokenized)

In [ ]:
test_results = trainer.predict(test_tokenized)
print(test_results)

In [ ]:
import numpy as np

test_predictions = np.argmax(
    test_results.predictions,
    axis=-1
)

print("Number of predictions:", len(test_predictions))
print("First 10 predictions:", test_predictions[:10])

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

test_accuracy = accuracy_score(
    y_test_encoded,
    test_predictions
)

test_macro_f1 = f1_score(
    y_test_encoded,
    test_predictions,
    average="macro"
)

print(f"Test Accuracy : {test_accuracy:.4f}")
print(f"Test Macro F1 : {test_macro_f1:.4f}")

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    y_test_encoded,
    test_predictions,
    target_names=label_encoder.classes_,
    digits=4
)

print(report)

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(
    y_test_encoded,
    test_predictions
)

cm_df = pd.DataFrame(
    cm,
    index=label_encoder.classes_,
    columns=label_encoder.classes_
)

print(cm_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm_df,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("AraBERT Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
print(trainer.state.best_model_checkpoint)

In [ ]:
from pathlib import Path

drive_model_path = Path(
    "/content/drive/MyDrive/PFA_Categorisation/models/arabert"
)

drive_model_path.mkdir(parents=True, exist_ok=True)

print("Folder:", drive_model_path)

In [ ]:
trainer.save_model(str(drive_model_path))
tokenizer.save_pretrained(str(drive_model_path))

print("AraBERT saved permanently to Google Drive.")

In [ ]:
import os

for file in os.listdir(drive_model_path):
    print(file)

In [ ]:
!pip install -U huggingface_hub

In [ ]:
from pathlib import Path

model_folder = Path(
    "/content/drive/MyDrive/PFA_Categorisation/models/arabert"
)

print("Folder exists:", model_folder.exists())

if model_folder.exists():
    for file in model_folder.iterdir():
        print(file.name)

In [2]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_folder(
    folder_path="/content/drive/MyDrive/PFA_Categorisation/models/arabert",
    repo_id="salah-2005/arabic-news-arabert",
    repo_type="model",
)

print("Upload completed!")

In [5]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

model_id = "salah-2005/arabic-news-arabert"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

print("Model loaded successfully!")
print(model.config.id2label)

config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  541MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!
{0: 'اقتصاد', 1: 'ثقافة', 2: 'دولي', 3: 'رياضة', 4: 'سياسة', 5: 'مجتمع'}


In [10]:
import torch

# Use GPU if available
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()

# Example Arabic news article
text = """وتشجيع الاستثمار المحلي والأجنبي، وتحفيز المؤسسات الصغيرة والمتوسطة، إضافة إلى خلق فرص عمل جديدة للشباب. وتأتي هذه الخطة في إطار الجهود الرامية إلى تعزيز الإنتاج الوطني وتحسين مناخ الأعمال ومواجهة التحديات الاقتصادية التي تشهدها الأسواق المحلية والدولية.

وأوضح مسؤول حكومي، خلال ندوة صحفية عقدت بالعاصمة، أن البرنامج الجديد يرتكز على مجموعة من الإجراءات المالية والاستثمارية التي تهدف إلى تحسين أداء القطاعات الإنتاجية، وتسهيل حصول المقاولات على التمويل، وتطوير البنية التحتية الاقتصادية. وأضاف أن الحكومة تسعى إلى تحقيق نمو مستدام من خلال دعم الأنشطة الصناعية والتجارية والخدماتية، مع التركيز على القطاعات القادرة على توفير قيمة مضافة للاقتصاد الوطني.

وتتضمن الخطة تقديم تسهيلات مالية للمقاولات الصغيرة والمتوسطة، خاصة تلك التي تواجه صعوبات في الحصول على القروض البنكية. كما ستعمل الجهات المعنية على تبسيط الإجراءات الإدارية المرتبطة بإنشاء الشركات، وتقليص المدة اللازمة للحصول على التراخيص، بهدف تشجيع رواد الأعمال على إطلاق مشاريع جديدة وتوسيع أنشطتهم الاقتصادية.

وفي هذا السياق، أكدت الحكومة أن تحسين مناخ الاستثمار يشكل أحد المحاور الرئيسية للبرنامج، مشيرة إلى أن جذب رؤوس الأموال يتطلب توفير بيئة اقتصادية مستقرة، وتطوير الخدمات المالية، وتعزيز الشفافية في المعاملات التجارية. ومن المنتظر أن تشمل الإجراءات الجديدة تحفيز الاستثمارات في مجالات الصناعة الغذائية، والطاقة المتجددة، والتكنولوجيا، والخدمات الرقمية، إلى جانب دعم المشاريع التي تساهم في رفع الصادرات الوطنية.

كما تهدف الخطة إلى تعزيز القدرة التنافسية للمنتجات المحلية في الأسواق الخارجية، من خلال تشجيع الشركات على تحديث معداتها وتطوير أساليب الإنتاج وتحسين جودة السلع والخدمات. وترى الجهات الاقتصادية أن رفع الإنتاجية وتخفيض تكاليف الإنتاج يمكن أن يساعدا المؤسسات الوطنية على مواجهة المنافسة الدولية والاستفادة من الفرص المتاحة في الأسواق الإقليمية والعالمية.

ومن بين الإجراءات التي يجري العمل عليها، إطلاق برامج تكوين وتأهيل لفائدة الشباب الباحثين عن العمل، بالتعاون مع المؤسسات الاقتصادية ومراكز التدريب المهني. وستركز هذه البرامج على المهارات المطلوبة في سوق الشغل، بما في ذلك الصناعات الحديثة، والتجارة الإلكترونية، والخدمات الرقمية، وإدارة المشاريع. وتهدف هذه المبادرات إلى تقليص الفجوة بين التكوين واحتياجات المقاولات، وتحسين فرص إدماج الخريجين في الحياة المهنية.

وفي ما يتعلق بالقطاع الصناعي، تعتزم الحكومة دعم المشاريع الرامية إلى تطوير الإنتاج المحلي وتقليص الاعتماد على بعض الواردات. وسيشمل الدعم تحديث الوحدات الصناعية، وتشجيع الابتكار، وتحسين سلاسل التوريد، وتطوير المناطق الصناعية. كما سيتم العمل على تسهيل وصول المستثمرين إلى العقارات المخصصة للأنشطة الاقتصادية، وتحسين الربط بين المناطق الصناعية والموانئ وشبكات النقل.

أما في القطاع الفلاحي، فتتضمن الخطة إجراءات لدعم الاستثمار في تقنيات الري الحديثة، وتحسين تخزين المنتجات الزراعية، وتطوير الصناعات الغذائية. ويهدف هذا التوجه إلى رفع مردودية الإنتاج، وتقليص الخسائر المرتبطة بالتخزين والنقل، وتعزيز القيمة المضافة للمنتجات المحلية. كما تسعى الحكومة إلى تشجيع التعاونيات والمقاولات الفلاحية على اعتماد أساليب إنتاج أكثر كفاءة واستدامة.

وفي المجال التجاري، ستعمل الجهات المختصة على دعم تحديث الأسواق وتطوير التجارة الإلكترونية، مع توفير برامج مواكبة للمقاولات الراغبة في تسويق منتجاتها عبر المنصات الرقمية. ويأتي ذلك في ظل التحولات التي يعرفها سلوك المستهلكين وتزايد أهمية الخدمات الرقمية في المعاملات التجارية. كما سيتم تشجيع المؤسسات على اعتماد وسائل الأداء الإلكتروني وتطوير خدمات التوصيل والتوزيع.

من جهة أخرى، أكدت الحكومة أن البرنامج الاقتصادي يتضمن إجراءات لتحسين تدبير المالية العمومية وتعزيز فعالية الإنفاق، مع الحرص على توجيه الموارد نحو المشاريع ذات الأثر الاقتصادي والاجتماعي. وستتم متابعة تنفيذ الاستثمارات العمومية وفق مؤشرات تتعلق بمعدلات الإنجاز، وفرص العمل المحدثةق.
"""

# 1. Convert text into tokens
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}

# 2. Get model predictions
with torch.no_grad():
    outputs = model(**inputs)

# 3. Convert scores to probabilities
probabilities = torch.softmax(
    outputs.logits,
    dim=-1
)[0]

# 4. Find the predicted category
predicted_id = probabilities.argmax().item()
predicted_category = model.config.id2label[predicted_id]

print("Predicted category:", predicted_category)
print(
    "Confidence:",
    f"{probabilities[predicted_id].item():.2%}"
)

print("\nProbabilities for all categories:")
for i, probability in enumerate(probabilities):
    category = model.config.id2label[i]
    print(f"{category}: {probability.item():.2%}")

Predicted category: اقتصاد
Confidence: 90.78%

Probabilities for all categories:
اقتصاد: 90.78%
ثقافة: 0.16%
دولي: 8.31%
رياضة: 0.06%
سياسة: 0.53%
مجتمع: 0.17%


In [7]:

import torch

# The model and tokenizer are already loaded.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)
model.eval()


def predict_article(text: str):
    # 1. Tokenize the user's text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # 2. Move inputs to the model's device
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # 3. Run the model
    with torch.no_grad():
        outputs = model(**inputs)

    # 4. Convert scores to probabilities
    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]

    # 5. Find the predicted category
    predicted_id = probabilities.argmax().item()
    predicted_category = model.config.id2label[
        predicted_id
    ]

    # 6. Prepare the response
    return {
        "category": predicted_category,
        "confidence": float(
            probabilities[predicted_id].item()
        ),
        "probabilities": {
            model.config.id2label[i]: float(probability)
            for i, probability in enumerate(probabilities)
        }
    }

In [14]:
text="""بدأ المنتخب الوطني لكرة القدم استعداداته لمواجهة حاسمة ضمن التصفيات المؤهلة إلى البطولة القارية، حيث خاض اللاعبون أول حصة تدريبية وسط أجواء من التركيز والحماس. وشهدت التدريبات مشاركة عدد من اللاعبين المحترفين في الخارج، إلى جانب عناصر تنشط في البطولة المحلية.

وأكد مدرب المنتخب، خلال ندوة صحفية، أن المباراة المقبلة تكتسي أهمية كبيرة في مسار التأهل، مشيراً إلى أن الطاقم التقني يعمل على تجهيز اللاعبين بدنياً وتكتيكياً لمواجهة المنافس. وأضاف أن الفريق سيحاول فرض أسلوبه منذ الدقائق الأولى، مع التركيز على الاستحواذ على الكرة والضغط على دفاع الخصم.

وشملت الحصة التدريبية تمارين خاصة بالتمرير السريع والتحرك دون كرة، إضافة إلى تدريبات على التسديد من خارج منطقة الجزاء وتنفيذ الكرات الثابتة. كما خصص الطاقم الطبي جزءاً من الحصة لمتابعة الحالة البدنية لبعض اللاعبين الذين عادوا مؤخراً من الإصابة.

ومن المنتظر أن يخوض المنتخب مباراة ودية قبل اللقاء الرسمي، بهدف اختبار بعض الخطط التكتيكية ومنح الفرصة للاعبين الجدد لإثبات قدراتهم. وأوضح المدرب أن اختيار التشكيلة الأساسية سيعتمد على جاهزية اللاعبين ومستواهم خلال التدريبات، مؤكداً أن جميع العناصر مطالبة بتقديم أداء قوي لخدمة المجموعة.

من جانبهم، عبّر عدد من اللاعبين عن استعدادهم للمباراة، مؤكدين أن الهدف هو تحقيق نتيجة إيجابية وإسعاد الجماهير. كما دعا قائد المنتخب المشجعين إلى مواصلة دعم الفريق، مشيراً إلى أن حضورهم في المدرجات يمنح اللاعبين حافزاً إضافياً خلال المنافسات الدولية.

وتترقب الجماهير الإعلان عن القائمة النهائية التي ستخوض المباراة، في وقت تتواصل فيه الاستعدادات التنظيمية لاستقبال اللقاء. ويسعى المنتخب إلى تحقيق الفوز وحصد النقاط الثلاث، من أجل تعزيز موقعه في ترتيب المجموعة والاقتراب خطوة إضافية من التأهل."""
result = predict_article(text)
print(result)

{'category': 'دولي', 'confidence': 0.839124858379364, 'probabilities': {'اقتصاد': 0.011546744965016842, 'ثقافة': 0.005290997214615345, 'دولي': 0.839124858379364, 'رياضة': 0.13314342498779297, 'سياسة': 0.0053028021939098835, 'مجتمع': 0.005591226741671562}}


In [10]:

# Six Arabic test articles
test_articles = [
    {
        "expected": "اقتصاد",
        "text": """أعلنت وزارة الاقتصاد عن إطلاق برنامج جديد لدعم المقاولات الصغيرة والمتوسطة، بهدف تشجيع الاستثمار وخلق فرص عمل إضافية. ويتضمن البرنامج تسهيلات للحصول على التمويل، إلى جانب مواكبة تقنية وإدارية لأصحاب المشاريع. وأكد مسؤولون أن المبادرة تسعى إلى تعزيز الإنتاج الوطني وتحسين مساهمة الشركات الصغرى في النمو الاقتصادي."""
    },
    {
        "expected": "ثقافة",
        "text": """احتضنت مدينة فاس فعاليات مهرجان ثقافي جمع فنانين وكتاباً من مختلف مناطق المغرب. وتضمن البرنامج معارض للفنون التشكيلية وندوات حول الأدب المغربي وعروضاً موسيقية تقليدية. وأشار المنظمون إلى أن هذه التظاهرة تهدف إلى التعريف بالتراث المحلي وتشجيع الشباب على المشاركة في الأنشطة الثقافية."""
    },
    {
        "expected": "دولي",
        "text": """عقد قادة عدد من الدول اجتماعاً دولياً لمناقشة التحديات الاقتصادية والأمنية التي تواجه المنطقة. وركزت المباحثات على تعزيز التعاون بين الحكومات وتطوير آليات مشتركة لمعالجة الأزمات الإنسانية. كما دعا المشاركون إلى مواصلة الحوار الدبلوماسي والعمل على إيجاد حلول سلمية للقضايا المطروحة."""
    },
    {
        "expected": "رياضة",
        "text": """حقق المنتخب الوطني لكرة القدم فوزاً مهماً في المباراة التي جمعته بمنتخب منافس ضمن التصفيات القارية. وسجل اللاعبون هدفين خلال الشوط الثاني، وسط حضور جماهيري كبير. وأشاد المدرب بأداء المجموعة والانضباط التكتيكي، مؤكداً أن الفريق سيواصل الاستعداد للمباراة المقبلة من أجل حجز بطاقة التأهل."""
    },
    {
        "expected": "سياسة",
        "text": """ناقش أعضاء البرلمان مشروع قانون جديداً خلال جلسة عامة خصصت لدراسة الإصلاحات المقترحة. وتبادل ممثلو الفرق البرلمانية الآراء بشأن بنود المشروع وآثاره على المؤسسات العمومية. وأكدت الحكومة أن النص يندرج ضمن برنامجها التشريعي، فيما طالبت بعض الفرق بتوضيحات إضافية قبل استكمال مناقشته."""
    },
    {
        "expected": "مجتمع",
        "text": """أطلقت جمعية محلية حملة تضامنية لفائدة الأسر ذات الدخل المحدود، بهدف توفير المواد الغذائية واللوازم المدرسية للأطفال. وشملت المبادرة عدداً من الأحياء، بمشاركة متطوعين وسكان المنطقة. وأوضح المنظمون أن الحملة تهدف إلى دعم الفئات المحتاجة وتعزيز روح التعاون والتكافل داخل المجتمع."""
    }
]

# Run your existing prediction function on each article
results = []

for i, article in enumerate(test_articles, start=1):
    prediction = predict_article(article["text"])

    expected = article["expected"]
    predicted = prediction["category"]
    confidence = prediction["confidence"]

    results.append({
        "article": i,
        "expected": expected,
        "predicted": predicted,
        "confidence": confidence,
        "correct": expected == predicted
    })

    print(f"\n--- Article {i} ---")
    print(f"Expected:   {expected}")
    print(f"Predicted:  {predicted}")
    print(f"Confidence: {confidence:.2%}")
    print("Result:", "CORRECT" if expected == predicted else "INCORRECT")

    print("\nProbabilities:")
    for category, probability in prediction["probabilities"].items():
        print(f"  {category}: {probability:.2%}")

# Summary
correct = sum(result["correct"] for result in results)

print("\n========== SUMMARY ==========")
print(f"Correct: {correct}/{len(results)}")
print(f"Accuracy: {correct / len(results):.2%}")


--- Article 1 ---
Expected:   اقتصاد
Predicted:  اقتصاد
Confidence: 99.81%
Result: CORRECT

Probabilities:
  اقتصاد: 99.81%
  ثقافة: 0.02%
  دولي: 0.07%
  رياضة: 0.01%
  سياسة: 0.06%
  مجتمع: 0.03%

--- Article 2 ---
Expected:   ثقافة
Predicted:  ثقافة
Confidence: 99.96%
Result: CORRECT

Probabilities:
  اقتصاد: 0.01%
  ثقافة: 99.96%
  دولي: 0.01%
  رياضة: 0.00%
  سياسة: 0.01%
  مجتمع: 0.01%

--- Article 3 ---
Expected:   دولي
Predicted:  دولي
Confidence: 97.25%
Result: CORRECT

Probabilities:
  اقتصاد: 0.11%
  ثقافة: 0.03%
  دولي: 97.25%
  رياضة: 0.03%
  سياسة: 2.52%
  مجتمع: 0.06%

--- Article 4 ---
Expected:   رياضة
Predicted:  رياضة
Confidence: 71.62%
Result: CORRECT

Probabilities:
  اقتصاد: 1.08%
  ثقافة: 0.48%
  دولي: 25.81%
  رياضة: 71.62%
  سياسة: 0.65%
  مجتمع: 0.37%

--- Article 5 ---
Expected:   سياسة
Predicted:  دولي
Confidence: 99.48%
Result: INCORRECT

Probabilities:
  اقتصاد: 0.03%
  ثقافة: 0.01%
  دولي: 99.48%
  رياضة: 0.02%
  سياسة: 0.43%
  مجتمع: 0.03%

--- Article 

In [2]:
!ls -la /content/arabic-news-classification

ls: cannot access '/content/arabic-news-classification': No such file or directory


In [3]:
!pwd
!ls -la

/content
total 16
drwxr-xr-x 1 root root 4096 Sep  4 13:32 .
drwxr-xr-x 1 root root 4096 Sep 18 17:21 ..
drwxr-xr-x 4 root root 4096 Sep  4 13:32 .config
drwxr-xr-x 1 root root 4096 Sep  4 13:32 sample_data


In [4]:
!find /content/arabic-news-classification -maxdepth 2 -type f

find: ‘/content/arabic-news-classification’: No such file or directory
